# Bước 3 — Dataset, transforms, DataLoader và metric

Notebook minh họa được đưa lên Git để các thành viên chạy chung. Logic nằm trong module Python; notebook giải thích và xem hình. Chọn kernel đã cài `requirements-training.txt`, rồi **Restart Kernel → Run All**.

Đầu vào: dữ liệu và split đã chốt ở bước 2. Không tải dữ liệu, không chia lại split, không train mô hình. Chỉ đọc train/validation; test và ClinicDB để đánh giá cuối. Không bắt buộc GPU để chạy notebook này.

## 1. Tìm gốc repo và đọc cấu hình

Không dùng đường dẫn ổ đĩa của một máy. `configs/unet.yaml` chứa preprocessing và thông số khởi đầu M1; các mục training chỉ là kế hoạch cho bước 4, chưa có vòng train hoạt động.


In [ ]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (start, *start.parents)
                     if (p / "src/datasets").is_dir() and (p / "README.md").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Mở notebook trong cây thư mục repo.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from PIL import Image
from IPython.display import display
from src.utils import load_config, seed_everything
from src.datasets.dataset import create_dataloaders
from src.datasets.preview import save_transform_preview
from src.metrics import binary_metrics, metrics_at_original_size

config = load_config(PROJECT_ROOT / "configs/unet.yaml")
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Cấu hình input:", config["data"]["image_size"])


## 2. Đọc một batch từ split cố định — giữ tỷ lệ bằng padding

Dataset đọc đúng ID, ảnh RGB và mask grayscale. Mask nhị phân hóa >127. Ảnh resize bilinear, mask nearest-neighbor **giữ cùng tỷ lệ** sao cho vừa khung 256×256; thêm padding để đặt ảnh ở giữa. Chênh lệch trái/phải và trên/dưới không quá 1 pixel; pixel dư đặt ở phải/dưới. Ví dụ 800×400 (rộng×cao) → 256×128 → thêm 64 pixel trên và dưới.

Padding ảnh màu đen trước ImageNet normalization, mask bằng 0. `valid_mask` kiểu bool, shape [B,1,H,W], chỉ True ở vùng nội dung; vùng viền không phải mô thật. Khi viết loss ở bước 4, phải dùng valid_mask để bỏ các pixel này khỏi phép tính loss.

Mỗi mẫu có `geometry`: original_size, source_size (sau augmentation), resized_size, input_size theo (cao, rộng), padding theo (trái, trên, phải, dưới). Augmentation thực hiện đồng bộ trước letterbox. Train shuffle; validation không augmentation/shuffle và giữ mask thật cỡ gốc trong list vì mỗi ảnh có kích thước khác nhau. Mặc định num_workers=0 cho Windows/notebook. Không đổi seed chia tập khi thử seed huấn luyện.


In [ ]:
loaders = create_dataloaders(PROJECT_ROOT, config)
train_batch = next(iter(loaders["train"]))
val_batch = next(iter(loaders["val"]))
print("Số mẫu:", {name: len(loader.dataset) for name, loader in loaders.items()})
print("Ảnh train:", tuple(train_batch["image"].shape), train_batch["image"].dtype)
print("Mask train:", tuple(train_batch["mask"].shape), train_batch["mask"].unique().tolist())
print("ID đầu:", train_batch["image_id"][0])
print("Kích thước mask gốc validation:", [tuple(m.shape) for m in val_batch["original_mask"]])
assert torch.isfinite(train_batch["image"]).all()
assert set(train_batch["mask"].unique().tolist()) <= {0, 1}

print("Geometry validation đầu tiên:", val_batch["geometry"][0])
print("Shape valid_mask:", tuple(train_batch["valid_mask"].shape))
print("Pixel ảnh thật mỗi mẫu:", train_batch["valid_mask"].flatten(1).sum(1).tolist())
assert not train_batch["mask"][~train_batch["valid_mask"]].any()
for geometry in val_batch["geometry"]:
    left, top, right, bottom = geometry["padding"]
    assert abs(left - right) <= 1 and abs(top - bottom) <= 1


## 3. Xem cùng một ảnh trước và sau letterbox

Bốn cột: ảnh gốc, overlay nhãn gốc, resize giữ tỷ lệ + padding (eval), và augmentation + padding (train). Khung xanh chỉ ranh giới nội dung, bên ngoài là padding đen. Ảnh nằm giữa khung vuông. Hình chỉ lấy ảnh train; cột eval minh họa xử lý cố định trên chính ảnh đó.

Lật ngang/dọc (mỗi phép p=0.5) và xoay góc 0/90/180/270° dùng cùng lựa chọn cho ảnh/mask, rồi mới resize và padding. Khi hiển thị, đảo normalization về RGB. Overlay dùng nhãn thật, không phải dự đoán mô hình. Ảnh/mask gốc không bị sửa; tensor sau xử lý ở RAM, chỉ hình minh họa được lưu.


In [ ]:
seed_everything(config["seed"])
figure_path = save_transform_preview(
    loaders["train"].dataset, PROJECT_ROOT / "results/figures/step3_transforms.png")
with Image.open(figure_path) as image:
    display(image.copy())
print("Hình:", figure_path.relative_to(PROJECT_ROOT))


## 4. Dice/IoU — kiểm tra bằng ví dụ biết trước

Metric tính riêng từng ảnh, sau đó lấy trung bình các ảnh. Hai mask rỗng cho Dice/IoU=1; chỉ một mask rỗng cho 0. Precision/Recall bằng 0 khi mẫu số không xác định. Đây là metric nhị phân, khác soft Dice dùng làm loss.

Ví dụ dưới: ảnh đầu có TP=1, số pixel dương dự đoán=2 và nhãn=2, nên Dice=0.5, IoU=1/3. Ảnh thứ hai cả hai rỗng nên Dice/IoU=1. **Đây là kiểm tra công thức, chưa phải kết quả của mô hình.**


In [ ]:
truth = torch.tensor([[[1., 1., 0.]], [[0., 0., 0.]]])
prediction = torch.tensor([[[1., 0., 1.]], [[0., 0., 0.]]])
scores = binary_metrics(prediction, truth, from_logits=False)
print({name: value.tolist() for name, value in scores.items()})
print("Dice trung bình hai ảnh:", scores["dice"].mean().item())
assert torch.allclose(scores["dice"], torch.tensor([0.5, 1.]))


## 5. Đánh giá đúng kích thước gốc sau padding

Logits mô hình nằm trong khung đầu vào có padding. `metrics_at_original_size` nhận cả `val_batch["geometry"]` để thực hiện đúng thứ tự: **bỏ padding → resize logits phần nội dung về kích thước gốc → sigmoid → threshold 0.5 → so với mask thật gốc**. Nếu decoder xuất logits nhỏ hơn khung, hàm upsample về khung trước khi crop. Không resize cả phần viền vào ảnh gốc và không resize mask thật để lấy điểm chính.

Nếu tính metric ở khung đầu vào, dùng `binary_metrics(..., valid_mask=batch["valid_mask"])` để loại viền. Đoạn dưới dùng logits nền giả để kiểm tra luồng, chưa phải hiệu năng mô hình. Khi tổng hợp nhiều batch, cộng điểm từng ảnh rồi chia tổng số ảnh, không trung bình không trọng số các batch khác kích thước.


In [ ]:
dummy_logits = torch.full_like(val_batch["mask"], -20)
original_scores = metrics_at_original_size(
    dummy_logits, val_batch["original_mask"], val_batch["geometry"], threshold=config["evaluation"]["threshold"])
print("Số điểm metric từng ảnh:", {name: value.numel() for name, value in original_scores.items()})
assert original_scores["dice"].numel() == len(val_batch["image_id"])


## Tiếp theo

Từ terminal có thể kiểm tra lại bước 3 bằng `python -m scripts.preview_data --project-root .`; kết quả lưu ở results/tables/step3_check.json.

Bước 4 sẽ viết U-Net tự cài, BCE+Dice, vòng train/validation và học thử 4–8 ảnh. Notebook hiện tại không tạo checkpoint hoặc chạy train.
